# Transformer from Scratch — Exercise Notebook

A decoder-only transformer (GPT/LLaMA style) in PyTorch, with **7 exercises** to complete.

**Two ways to use this project:**

| Mode | Config | Device | How to run |
|------|--------|--------|------------|
| **Local / CI** | `ModelConfig.tiny()` — 64-dim, 2 layers | CPU | `pytest tests/ -v` |
| **Colab (this notebook)** | `ModelConfig.small()` — 256-dim, 4 layers | GPU | Run cells below |

### Exercise order

| # | Exercise | File | Test |
|---|----------|------|------|
| 1 | RMSNorm | `model.py` | `test_rmsnorm.py` |
| 2 | RoPE | `rope.py` | `test_rope.py` |
| 3 | Scaled dot-product attention | `model.py` | `test_attention.py` |
| 4 | Causal mask | `model.py` | `test_mask.py` |
| 5 | SwiGLU feed-forward | `model.py` | `test_feedforward.py` |
| 6 | KV cache | `model.py` | `test_kv_cache.py` |
| 7 | Top-k / top-p sampling | `generate.py` | `test_sampling.py` |

## 0 — Setup

> **Colab**: Go to *Runtime → Change runtime type → GPU* before running.

In [ ]:
!pip install -q torch pytest

In [ ]:
import torch, time

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    DEVICE = torch.device("cuda")
else:
    print("No GPU detected — running on CPU (tests still work, GPU sections will be skipped)")
    DEVICE = torch.device("cpu")

---
## Part 1 — Unit Tests (CPU, tiny config)

Run these as you implement each exercise. They use `ModelConfig.tiny()` and run on CPU.

### Exercise 1 — RMSNorm
Edit `model.py` → `RMSNorm.forward`

In [ ]:
!pytest tests/test_rmsnorm.py -v

### Exercise 2 — Rotary Position Embeddings (RoPE)
Edit `rope.py` → `precompute_rope_frequencies` + `apply_rope`

In [ ]:
!pytest tests/test_rope.py -v

### Exercise 3 — Scaled Dot-Product Attention
Edit `model.py` → `scaled_dot_product_attention`

In [ ]:
!pytest tests/test_attention.py -v

### Exercise 4 — Causal Mask
Edit `model.py` → `create_causal_mask`

In [ ]:
!pytest tests/test_mask.py -v

### Exercise 5 — SwiGLU Feed-Forward
Edit `model.py` → `FeedForward.forward`

In [ ]:
!pytest tests/test_feedforward.py -v

### Exercise 6 — KV Cache
Edit `model.py` → `Attention.update_kv_cache`

*Requires exercises 1–5 to be implemented.*

In [ ]:
!pytest tests/test_kv_cache.py -v

### Exercise 7 — Top-k / Top-p Sampling
Edit `generate.py` → `top_k_top_p_filtering`

In [ ]:
!pytest tests/test_sampling.py -v

### End-to-End (CPU, tiny)

*Requires all 7 exercises.*

In [ ]:
!pytest tests/test_e2e.py -v

---
## Part 2 — GPU Validation (larger model)

The unit tests above use `ModelConfig.tiny()` on CPU.
This section instantiates the **larger** `ModelConfig.small()` (256-dim, 4 layers, 8 heads)
on GPU and validates that everything works at scale.

> Skip this section if no GPU is available — Part 1 is sufficient to verify correctness.

In [ ]:
from config import ModelConfig
from model import Transformer
from generate import generate, generate_text
from tokenizer import CharTokenizer

config = ModelConfig.small()
print("Model config:")
print(f"  dim={config.dim}, layers={config.n_layers}, heads={config.n_heads}, "
      f"kv_heads={config.n_kv_heads}, vocab={config.vocab_size}")

torch.manual_seed(42)
model = Transformer(config).to(DEVICE)
model.eval()

param_count = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {param_count:,} ({param_count / 1e6:.2f}M)")
print(f"  Device: {DEVICE}")

### GPU — Forward pass sanity check

In [ ]:
batch, seq_len = 4, 128
tokens = torch.randint(0, config.vocab_size, (batch, seq_len), device=DEVICE)

with torch.no_grad():
    logits = model(tokens)

print(f"Input shape:  {tokens.shape}")
print(f"Output shape: {logits.shape}")
assert logits.shape == (batch, seq_len, config.vocab_size), "Shape mismatch!"
print("Forward pass OK")

### GPU — KV cache consistency check

Verifies that cached generation produces the same logits as full-sequence forward.

In [ ]:
seq_len = 64
tokens = torch.randint(0, config.vocab_size, (1, seq_len), device=DEVICE)

with torch.no_grad():
    logits_full = model(tokens, start_pos=0, use_cache=False)

    model.init_cache(1, DEVICE)
    logits_prefill = model(tokens[:, :32], start_pos=0, use_cache=True)

    incremental_logits = []
    for i in range(32, seq_len):
        step_logits = model(tokens[:, i:i+1], start_pos=i, use_cache=True)
        incremental_logits.append(step_logits)
    model.reset_cache()

logits_incremental = torch.cat(incremental_logits, dim=1)

prefill_ok = torch.allclose(logits_full[:, :32], logits_prefill, atol=1e-4)
decode_ok = torch.allclose(logits_full[:, 32:], logits_incremental, atol=1e-4)

print(f"Prefill matches full forward:      {prefill_ok}")
print(f"Incremental decode matches:         {decode_ok}")
if prefill_ok and decode_ok:
    print("KV cache consistency OK")
else:
    max_diff_prefill = (logits_full[:, :32] - logits_prefill).abs().max().item()
    max_diff_decode = (logits_full[:, 32:] - logits_incremental).abs().max().item()
    print(f"Max diff prefill: {max_diff_prefill:.6f}, decode: {max_diff_decode:.6f}")

### GPU — Generation throughput benchmark

Measures tokens/sec for prefill and decode phases.

In [ ]:
prompt_len = 64
gen_tokens = 128
prompt = torch.randint(0, config.vocab_size, (1, prompt_len), device=DEVICE)

if DEVICE.type == "cuda":
    torch.cuda.synchronize()

t0 = time.perf_counter()
output = generate(model, prompt, max_new_tokens=gen_tokens, temperature=0.8, top_k=40)
if DEVICE.type == "cuda":
    torch.cuda.synchronize()
t1 = time.perf_counter()

elapsed = t1 - t0
total_tokens = output.shape[1]
print(f"Prompt: {prompt_len} tokens → Generated: {gen_tokens} tokens")
print(f"Total time:   {elapsed:.3f}s")
print(f"Throughput:   {gen_tokens / elapsed:.1f} tokens/sec (decode)")
print(f"Output shape: {output.shape}")

### GPU — Text generation demo

The model has random weights, so output will be gibberish — that's expected.
The point is to verify the full pipeline runs end-to-end on GPU.

In [ ]:
tokenizer = CharTokenizer(vocab_size=config.vocab_size)

for temp in [0.0, 0.5, 1.0]:
    text = generate_text(
        model, tokenizer, "The transformer is",
        max_new_tokens=80, temperature=temp, top_k=40, top_p=0.9,
    )
    label = "greedy" if temp == 0 else f"temp={temp}"
    print(f"[{label}] {text[:120]}")
    print()

### GPU — Memory profile

Shows how much GPU memory the model and KV cache consume.

In [ ]:
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

    prompt = torch.randint(0, config.vocab_size, (1, 128), device=DEVICE)
    _ = generate(model, prompt, max_new_tokens=64, temperature=0)

    allocated = torch.cuda.max_memory_allocated() / 1e6
    reserved = torch.cuda.max_memory_reserved() / 1e6
    print(f"Peak memory allocated: {allocated:.1f} MB")
    print(f"Peak memory reserved:  {reserved:.1f} MB")
else:
    print("GPU memory profiling skipped (no CUDA device)")

---
## Part 3 — End-to-End: Training + Inference

Train the model on a small built-in text corpus, then run real inference.
No external data download needed — the training text is embedded in `train.py`.

> **Colab GPU tip**: with a T4 GPU and `ModelConfig.small()`, 1000 steps takes ~20s.
> On CPU with `ModelConfig.tiny()`, 300 steps takes ~10s.

### Step 1 — Choose config based on device

In [ ]:
from config import ModelConfig
from train import TrainConfig, train, load_checkpoint, TINY_TEXT
from generate import generate_text
from tokenizer import CharTokenizer
import torch

# Auto-detect device
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    # Larger model + more steps for GPU
    model_cfg = ModelConfig.small()
    train_cfg = TrainConfig(
        batch_size=16,
        max_steps=1000,
        seq_len=128,
        log_every=100,
        eval_every=200,
        device="cuda",
    )
else:
    DEVICE = "cpu"
    print("No GPU — using tiny config on CPU")
    # Tiny model for CPU
    model_cfg = ModelConfig.tiny()
    train_cfg = TrainConfig(
        batch_size=4,
        max_steps=300,
        seq_len=64,
        log_every=50,
        eval_every=100,
        device="cpu",
    )

print(f"Model: dim={model_cfg.dim}, layers={model_cfg.n_layers}, heads={model_cfg.n_heads}")
print(f"Training: {train_cfg.max_steps} steps, batch={train_cfg.batch_size}, seq_len={train_cfg.seq_len}")

### Step 2 — Train

The built-in training text is ~2700 tokens of text about transformers.
Watch the loss go down and the sample quality improve every `eval_every` steps.

In [ ]:
model, tokenizer = train(model_cfg, train_cfg, text=TINY_TEXT)

### Step 3 — Inspect the loss curve

In [ ]:
try:
    import matplotlib.pyplot as plt
    from train import TrainConfig
    # Re-run a short training to capture losses for plotting
    # (skip if you just want inference)
    print("matplotlib available — you can plot the loss curve")
    print("To plot: collect losses from train() and call plt.plot(losses)")
except ImportError:
    print("matplotlib not installed — run: !pip install matplotlib")

### Step 4 — Inference with the trained model

The model is trained on text about transformers, so prompts related to that topic
tend to produce the most coherent continuations.

In [ ]:
model.eval()
prompts = ["The transformer", "Attention is", "The key", "Training uses"]

print("=== Greedy (temperature=0) ===")
for p in prompts:
    out = generate_text(model, tokenizer, p, max_new_tokens=80, temperature=0)
    print(f"[{p!r}]\n  {out!r}\n")

print("=== Sampling (temperature=0.8, top_k=20) ===")
for p in prompts:
    out = generate_text(model, tokenizer, p, max_new_tokens=80, temperature=0.8, top_k=20)
    print(f"[{p!r}]\n  {out!r}\n")

### Step 5 — Save & reload checkpoint

The checkpoint is saved automatically at the end of `train()`.
This cell shows how to reload it in a new session.

In [ ]:
# Reload model from checkpoint (simulates a new session)
reloaded_model, reloaded_cfg = load_checkpoint("checkpoint.pt", device=DEVICE)
tokenizer2 = CharTokenizer(vocab_size=reloaded_cfg.vocab_size)

out = generate_text(reloaded_model, tokenizer2, "The transformer", max_new_tokens=100, temperature=0.8, top_k=20)
print("Reloaded model output:")
print(out)

### (Optional) Step 6 — Train on your own text

Replace `my_text` with any string you want — a book excerpt, song lyrics, code, etc.

In [ ]:
my_text = """
To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them.
""".strip() * 20  # repeat to get enough tokens

custom_model, custom_tokenizer = train(
    model_cfg=model_cfg,
    train_cfg=TrainConfig(
        batch_size=train_cfg.batch_size,
        max_steps=train_cfg.max_steps,
        seq_len=train_cfg.seq_len,
        log_every=train_cfg.log_every,
        eval_every=train_cfg.eval_every,
        checkpoint_path="checkpoint_custom.pt",
        device=DEVICE,
    ),
    text=my_text,
)

print("\n--- Custom text inference ---")
out = generate_text(custom_model, custom_tokenizer, "To be", max_new_tokens=100, temperature=0.8, top_k=20)
print(out)